![Two data scientists working on a dashboard.](hr-image-small.png)

A common problem when creating models to generate business value from data is that the datasets can be so large that it can take days for the model to generate predictions. Ensuring that your dataset is stored as efficiently as possible is crucial for allowing these models to run on a more reasonable timescale without having to reduce the size of the dataset.

You've been hired by a major online data science training provider called *Training Data Ltd.* to clean up one of their largest customer datasets. This dataset will eventually be used to predict whether their students are looking for a new job or not, information that they will then use to direct them to prospective recruiters.

You've been given access to `customer_train.csv`, which is a subset of their entire customer dataset, so you can create a proof-of-concept of a much more efficient storage solution. The dataset contains anonymized student information, and whether they were looking for a new job or not during training:

| Column                   | Description                                                                      |
|------------------------- |--------------------------------------------------------------------------------- |
| `student_id`             | A unique ID for each student.                                                    |
| `city`                   | A code for the city the student lives in.                                        |
| `city_development_index` | A scaled development index for the city.                                         |
| `gender`                 | The student's gender.                                                            |
| `relevant_experience`    | An indicator of the student's work relevant experience.                          |
| `enrolled_university`    | The type of university course enrolled in (if any).                              |
| `education_level`        | The student's education level.                                                   |
| `major_discipline`       | The educational discipline of the student.                                       |
| `experience`             | The student's total work experience (in years).                                  |
| `company_size`           | The number of employees at the student's current employer.                       |
| `company_type`           | The type of company employing the student.                                       |
| `last_new_job`           | The number of years between the student's current and previous jobs.             |
| `training_hours`         | The number of hours of training completed.                                       |
| `job_change`             | An indicator of whether the student is looking for a new job (`1`) or not (`0`). |

In [1]:
# Import necessary libraries
import pandas as pd

# Load the dataset
ds_jobs = pd.read_csv("customer_train.csv")

# View the dataset
ds_jobs.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


In [2]:
# Create a copy of ds_jobs for transforming
ds_jobs_transformed = ds_jobs.copy()

# Start coding here. Use as many cells as you like!

In [3]:
# explore the data to understand the contents and data types of the values in each column
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              19158 non-null  int64  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float64
 3   gender                  14650 non-null  object 
 4   relevant_experience     19158 non-null  object 
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int64  
 13  job_change              19158 non-null  float64
dtypes: float64(2), int64(2), object(10)
me

In [4]:
ds_jobs_transformed['city'].value_counts()

city_103    4355
city_21     2702
city_16     1533
city_114    1336
city_160     845
            ... 
city_121       3
city_129       3
city_111       3
city_171       1
city_140       1
Name: city, Length: 123, dtype: int64

In [5]:
len(ds_jobs_transformed['city'].unique())

123

In [6]:
# find all non-numeric columns that have exactly two unique values, so we can convert them to category
two_level_cats = [ col for col in ds_jobs_transformed.columns if ds_jobs_transformed[col].dtype == "object" \
                  or pd.api.types.is_bool_dtype(ds_jobs_transformed[col]) \
                  if ds_jobs_transformed[col].nunique(dropna=False) == 2]

In [7]:
two_level_cats

['relevant_experience']

In [8]:
# Create map to map values to True or False
relevant_experience_map = { 'Has relevant experience': True, 'No relevant experience': False }

In [9]:
ds_jobs_transformed['relevant_experience'] = ds_jobs_transformed['relevant_experience'].map(relevant_experience_map)

In [10]:
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              19158 non-null  int64  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float64
 3   gender                  14650 non-null  object 
 4   relevant_experience     19158 non-null  bool   
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int64  
 13  job_change              19158 non-null  float64
dtypes: bool(1), float64(2), int64(2), obje

In [11]:
job_change_map = { 0.0 : False, 1.0 : True }
ds_jobs_transformed['job_change'] = ds_jobs_transformed['job_change'].map(job_change_map)
ds_jobs_transformed.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,True,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,True
1,29725,city_40,0.776,Male,False,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,False
2,11561,city_21,0.624,NaN,False,Full time course,Graduate,STEM,5,NaN,NaN,never,83,False
3,33241,city_115,0.789,NaN,False,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,True
4,666,city_162,0.767,Male,True,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,False


In [12]:
# Columns containing integers only must be stored as 32-bit integers (int32).
int_cols = ds_jobs_transformed.select_dtypes(include='int64').columns
ds_jobs_transformed[int_cols] = ds_jobs_transformed[int_cols].astype('int32')
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              19158 non-null  int32  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float64
 3   gender                  14650 non-null  object 
 4   relevant_experience     19158 non-null  bool   
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int32  
 13  job_change              19158 non-null  bool   
dtypes: bool(2), float64(1), int32(2), obje

In [13]:
float_cols = ds_jobs_transformed.select_dtypes(include='float64').columns
ds_jobs_transformed[float_cols] = ds_jobs_transformed[float_cols].astype('float16')
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              19158 non-null  int32  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float16
 3   gender                  14650 non-null  object 
 4   relevant_experience     19158 non-null  bool   
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int32  
 13  job_change              19158 non-null  bool   
dtypes: bool(2), float16(1), int32(2), obje

In [14]:
ds_jobs_transformed['city'] = ds_jobs_transformed['city'].astype('category')

In [15]:
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   student_id              19158 non-null  int32   
 1   city                    19158 non-null  category
 2   city_development_index  19158 non-null  float16 
 3   gender                  14650 non-null  object  
 4   relevant_experience     19158 non-null  bool    
 5   enrolled_university     18772 non-null  object  
 6   education_level         18698 non-null  object  
 7   major_discipline        16345 non-null  object  
 8   experience              19093 non-null  object  
 9   company_size            13220 non-null  object  
 10  company_type            13018 non-null  object  
 11  last_new_job            18735 non-null  object  
 12  training_hours          19158 non-null  int32   
 13  job_change              19158 non-null  bool    
dtypes: bool(2), category(1

In [16]:
ds_jobs_transformed['city'].cat.categories

Index(['city_1', 'city_10', 'city_100', 'city_101', 'city_102', 'city_103',
       'city_104', 'city_105', 'city_106', 'city_107',
       ...
       'city_84', 'city_89', 'city_9', 'city_90', 'city_91', 'city_93',
       'city_94', 'city_97', 'city_98', 'city_99'],
      dtype='object', length=123)

In [17]:
ds_jobs_transformed['gender'].value_counts()

Male      13221
Female     1238
Other       191
Name: gender, dtype: int64

In [18]:
ds_jobs_transformed['enrolled_university'].value_counts()

no_enrollment       13817
Full time course     3757
Part time course     1198
Name: enrolled_university, dtype: int64

In [19]:
ds_jobs_transformed['education_level'].value_counts()

Graduate          11598
Masters            4361
High School        2017
Phd                 414
Primary School      308
Name: education_level, dtype: int64

In [20]:
ds_jobs_transformed['major_discipline'].value_counts()

STEM               14492
Humanities           669
Other                381
Business Degree      327
Arts                 253
No Major             223
Name: major_discipline, dtype: int64

In [21]:
ds_jobs_transformed['company_size'].value_counts()

50-99        3083
100-499      2571
10000+       2019
10-49        1471
1000-4999    1328
<10          1308
500-999       877
5000-9999     563
Name: company_size, dtype: int64

In [22]:
ds_jobs_transformed['company_type'].value_counts()

Pvt Ltd                9817
Funded Startup         1001
Public Sector           955
Early Stage Startup     603
NGO                     521
Other                   121
Name: company_type, dtype: int64

In [23]:
ds_jobs_transformed['last_new_job'].value_counts()

1        8040
>4       3290
2        2900
never    2452
4        1029
3        1024
Name: last_new_job, dtype: int64

In [24]:
import pandas as pd
from pandas.api.types import CategoricalDtype

def make_categorical(df, col, fill_value=None, ordered_categories=None):
    """
    Convert a column to categorical with optional missing-value replacement
    and optional ordering.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame.
    col : str
        Column name to convert.
    fill_value : optional
        Value to replace NaNs with. If None, NaNs are kept.
    ordered_categories : list, optional
        If provided, defines the category order and makes the dtype ordered.
        If None, categories are inferred and unordered.

    Returns
    -------
    pandas.DataFrame
        A copy of the DataFrame with the column converted.
    """
    df = df.copy()

    # Handle missing values
    if fill_value is not None:
        df[col] = df[col].fillna(fill_value)

    # If user provides an ordering
    if ordered_categories is not None:
        cat_type = CategoricalDtype(categories=ordered_categories, ordered=True)
        df[col] = df[col].astype(cat_type)
    else:
        # Unordered categorical
        df[col] = df[col].astype("category")

    return df

In [25]:
# Process 'gender' : 'Male', 'Female', 'Unknown' 
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='gender', fill_value='Unknown', ordered_categories=['Unknown', 'Male', 'Female'])

In [26]:
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   student_id              19158 non-null  int32   
 1   city                    19158 non-null  category
 2   city_development_index  19158 non-null  float16 
 3   gender                  18967 non-null  category
 4   relevant_experience     19158 non-null  bool    
 5   enrolled_university     18772 non-null  object  
 6   education_level         18698 non-null  object  
 7   major_discipline        16345 non-null  object  
 8   experience              19093 non-null  object  
 9   company_size            13220 non-null  object  
 10  company_type            13018 non-null  object  
 11  last_new_job            18735 non-null  object  
 12  training_hours          19158 non-null  int32   
 13  job_change              19158 non-null  bool    
dtypes: bool(2), category(2

In [27]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='enrolled_university', fill_value='no_enrollment_info', ordered_categories=['no_enrollment_info', 'no_enrollment', 'Part time course','Full time course'])

In [28]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='education_level', fill_value='Unknown', \
                                       ordered_categories=['Unknown', 'Primary School', 'High School','Graduate', 'Masters', 'Phd'])

In [29]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='major_discipline', fill_value='No Major')

In [30]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='company_size', fill_value='Unknown', \
                                      ordered_categories=['Unknown', '<10', '10-49', '50-99', '100-499', '500-999', '1000-4999', '5000-9999', '10000+'])

In [31]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='company_type', fill_value='Unknown')

In [32]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='last_new_job', fill_value='Unknown', \
                                      ordered_categories=['Unknown', 'never', '1', '2', '3', '4', '>4'])

In [33]:
ds_jobs_transformed['experience'].value_counts()

>20    3286
5      1430
4      1403
3      1354
6      1216
2      1127
7      1028
10      985
9       980
8       802
15      686
11      664
14      586
1       549
<1      522
16      508
12      494
13      399
17      342
19      304
18      280
20      148
Name: experience, dtype: int64

In [34]:
ds_jobs_transformed = make_categorical(ds_jobs_transformed, col='experience', fill_value='<1', \
                                      ordered_categories=['<1', '1', '2', '3', '4', '5', '6', '7', \
                                                          '8','9','10','11','12','13','14','15','16','17','18','19','20','>20'])

In [35]:
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   student_id              19158 non-null  int32   
 1   city                    19158 non-null  category
 2   city_development_index  19158 non-null  float16 
 3   gender                  18967 non-null  category
 4   relevant_experience     19158 non-null  bool    
 5   enrolled_university     19158 non-null  category
 6   education_level         19158 non-null  category
 7   major_discipline        19158 non-null  category
 8   experience              19158 non-null  category
 9   company_size            19158 non-null  category
 10  company_type            19158 non-null  category
 11  last_new_job            19158 non-null  category
 12  training_hours          19158 non-null  int32   
 13  job_change              19158 non-null  bool    
dtypes: bool(2), category(9

In [36]:
ds_jobs_transformed['experience'].value_counts()

>20    3286
5      1430
4      1403
3      1354
6      1216
2      1127
7      1028
10      985
9       980
8       802
15      686
11      664
<1      587
14      586
1       549
16      508
12      494
13      399
17      342
19      304
18      280
20      148
Name: experience, dtype: int64

In [37]:
ds_jobs_transformed['experience'].cat.categories

Index(['<1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', '15', '16', '17', '18', '19', '20', '>20'],
      dtype='object')

In [38]:
ds_jobs_transformed = ds_jobs_transformed[(ds_jobs_transformed['experience'] >= '10') & \
                                    (ds_jobs_transformed['company_size']  >= '1000-4999')] 

In [39]:
ds_jobs_transformed.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
9,699,city_103,0.919922,Unknown,True,no_enrollment,Graduate,STEM,17,10000+,Pvt Ltd,>4,123,False
12,25619,city_61,0.913086,Male,True,no_enrollment,Graduate,STEM,>20,1000-4999,Pvt Ltd,3,23,False
31,22293,city_103,0.919922,Male,True,Part time course,Graduate,STEM,19,5000-9999,Pvt Ltd,>4,141,False
34,26494,city_16,0.910156,Male,True,no_enrollment,Graduate,Business Degree,12,5000-9999,Pvt Ltd,3,145,False
40,2547,city_114,0.925781,Female,True,Full time course,Masters,STEM,16,1000-4999,Public Sector,2,14,False
